In [1]:
import pandas as pd

In [2]:
raw_holdout_file = "../data/raw/GDPa3_80 IgGs_full.xlsx"

In [3]:
file = pd.ExcelFile(raw_holdout_file)
file.sheet_names


['Definitions',
 'Sequences',
 'Assay Data - tidy format',
 'Assay Data - average',
 'Versioning']

In [4]:
sequence_df = pd.read_excel(file, "Sequences")
sequence_df.shape

(80, 7)

In [5]:
assays_df = pd.read_excel(file, "Assay Data - average")
assays_df.shape

(80, 34)

In [6]:
merged_holdout_df = sequence_df.merge(assays_df, on="antibody_id", how="left")
print(merged_holdout_df.head().to_markdown())

|    | antibody_id   | hc_subtype   | lc_subtype   | vh_protein_sequence                                                                                                | lc_protein_sequence                                                                                            | hc_dna_sequence                                                                                                                                                                                                                                                                                                                                        | lc_dna_sequence                                                                                                                                                                                                                                                                                                                            |   hac_rt_avg |   hac_rt_replicates |   

In [7]:
replicate_std_cols = [
    "hac_rt_replicates",
    "hac_rt_stddev",
    "hic_rt_replicates",
    "hic_rt_stddev",
    "polyreactivity_prscore_cho_replicates",
    "polyreactivity_prscore_cho_stddev",
    "polyreactivity_prscore_ova_replicates",
    "polyreactivity_prscore_ova_stddev",
    "sec_%monomer_replicates",
    "sec_%monomer_stddev",
    "smac_rt_replicates",
    "smac_rt_stddev",
    "titer_replicates",
    "titer_stddev",
    "tm1_nanodsf_replicates",
    "tm1_nanodsf_stddev",
    "tm2_nanodsf_replicates",
    "tm2_nanodsf_stddev",
    "tm3_nanodsf_replicates",
    "tm3_nanodsf_stddev",
    "tonset_nanodsf_replicates",
    "tonset_nanodsf_stddev",
]

drop_extra_assays = [
    "hac_rt_avg",
    "sec_%monomer_avg",
    "tm3_nanodsf_avg",
]

drop_dna = [
    "hc_dna_sequence",
    "lc_dna_sequence",
]

In [8]:
filtered_holdout_df = merged_holdout_df.drop(
    columns=(replicate_std_cols + drop_extra_assays + drop_dna), errors="ignore"
)

filtered_holdout_df.columns

Index(['antibody_id', 'hc_subtype', 'lc_subtype', 'vh_protein_sequence',
       'lc_protein_sequence', 'hic_rt_avg', 'polyreactivity_prscore_cho_avg',
       'polyreactivity_prscore_ova_avg', 'smac_rt_avg', 'titer_avg',
       'tm1_nanodsf_avg', 'tm2_nanodsf_avg', 'tonset_nanodsf_avg'],
      dtype='object')

In [9]:
filtered_holdout_df = filtered_holdout_df.rename(columns={
    "lc_protein_sequence": "vl_protein_sequence",
    "smac_rt_avg": "SMAC",
    "hic_rt_avg": "HIC",
    "polyreactivity_prscore_cho_avg": "PR_CHO",
    "polyreactivity_prscore_ova_avg": "PR_Ova",
    "tm1_nanodsf_avg": "Tm1",
    "tm2_nanodsf_avg": "Tm2",
    "tonset_nanodsf_avg": "Tonset",
    "titer_avg": "Titer",
})

print(filtered_holdout_df.head().to_markdown())

|    | antibody_id   | hc_subtype   | lc_subtype   | vh_protein_sequence                                                                                                | vl_protein_sequence                                                                                            |     HIC |    PR_CHO |    PR_Ova |    SMAC |   Titer |     Tm1 |     Tm2 |   Tonset |
|---:|:--------------|:-------------|:-------------|:-------------------------------------------------------------------------------------------------------------------|:---------------------------------------------------------------------------------------------------------------|--------:|----------:|----------:|--------:|--------:|--------:|--------:|---------:|
|  0 | GDPa3-001     | IgG1         | Lambda       | EVQLVESGGGLVQPGGSLRLSCATSGFTFNRYWMSWVRQAPGKGLEWVANIDEHGSEKNYADYVKGRFTISRDNATTSLFLQMNSLRVEDSALYYCTLGDYWGQGTLVTVSS   | SYELTQPSSVSVSPGQTATITCSGDVLAKRYARWFQQKPGQAPVLVIYKDSERPSGIPERFSGSSSGTTVTLTISGAQVEDEADYYCYSVPDNR

In [10]:
filtered_holdout_df.columns

Index(['antibody_id', 'hc_subtype', 'lc_subtype', 'vh_protein_sequence',
       'vl_protein_sequence', 'HIC', 'PR_CHO', 'PR_Ova', 'SMAC', 'Titer',
       'Tm1', 'Tm2', 'Tonset'],
      dtype='object')

## HOLDOUT DATA DOES NOT INCLUDE "AC-SINS_pH6.0" and "AC-SINS_pH7.4"

Can be considered that these are not in "real world data" may have to remove from training data if not available in holdout

Antibody name is also not in holdout. Can be removed from training, but will not have an effect on model.

In [11]:
if "antibody_name" not in filtered_holdout_df.columns:
    filtered_holdout_df["antibody_name"] = pd.NA

filtered_holdout_df["AC-SINS_pH6.0"] = pd.NA
filtered_holdout_df["AC-SINS_pH7.4"] = pd.NA

In [12]:
training_order = [
    "antibody_id",
    "antibody_name",
    "SMAC",
    "HIC",
    "PR_CHO",
    "PR_Ova",
    "AC-SINS_pH6.0",
    "AC-SINS_pH7.4",
    "Tonset",
    "Tm1",
    "Tm2",
    "vh_protein_sequence",
    "vl_protein_sequence",
    "hc_subtype",
    "lc_subtype",
    "Titer",
]

In [13]:
cleaned_holdout_df = filtered_holdout_df[training_order]

In [14]:
# For later dropping

# cleaned_holdout_df = cleaned_holdout_df.drop(columns=["antibody_name", "AC-SINS_pH6.0", "AC-SINS_pH7.4"])


In [15]:
cleaned_holdout_df.to_csv("../data/test_data/cleaned_holdout_data.csv")

In [16]:
print(cleaned_holdout_df.to_markdown())

|    | antibody_id   | antibody_name   |    SMAC |       HIC |      PR_CHO |     PR_Ova | AC-SINS_pH6.0   | AC-SINS_pH7.4   |   Tonset |     Tm1 |      Tm2 | vh_protein_sequence                                                                                                                  | vl_protein_sequence                                                                                               | hc_subtype   | lc_subtype   |      Titer |
|---:|:--------------|:----------------|--------:|----------:|------------:|-----------:|:----------------|:----------------|---------:|--------:|---------:|:-------------------------------------------------------------------------------------------------------------------------------------|:------------------------------------------------------------------------------------------------------------------|:-------------|:-------------|-----------:|
|  0 | GDPa3-001     | <NA>            | 2.60667 |   2.44172 | 0.0456681   | 0.0236293  | <NA>  